# ADNI Data Integration Pipeline
**Study:** APOE ε4 × Hippocampal Volume × Cognitive Decline

This notebook implements the full data pipeline addressing all reviewer comments:
- Expands sample from 133 → 2,386+ subjects
- Hippocampal volume as **continuous** (not median-split)
- Adds GDS (depression) and NPI-Q (neuropsychiatric) covariates
- Produces analysis-ready datasets saved to `reports/`

**Output files:**
- `ADNI_Master_Longitudinal.csv` — all visits, all subjects
- `ADNI_Complete_Cases.csv` — subjects with MMSE + APOE + Hippocampus + Dx
- `ADNI_Baseline_Analysis.csv` — baseline only (cross-sectional)
- `ADNI_Table1_Characteristics.csv` — summary stats by APOE dose
- `ADNI_Data_Quality_Report.txt` — quality metrics

## 0. Setup

In [59]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

# Paths
BASE_DIR  = '/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Multimodel_study'
TABLES    = os.path.join(BASE_DIR, 'tables')
REPORTS   = os.path.join(BASE_DIR, 'reports')
os.makedirs(REPORTS, exist_ok=True)

# Visit-code normalisation
BL_CODES = {'sc', 'scmri', 'f', 'bl'}

def norm_viscode(v):
    if pd.isna(v): return v
    v = str(v).lower().strip()
    return 'bl' if v in BL_CODES else v

print(f'Tables : {TABLES}')
print(f'Reports: {REPORTS}')
print(f'Started: {datetime.now():%Y-%m-%d %H:%M}')

Tables : /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Multimodel_study/tables
Reports: /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Multimodel_study/reports
Started: 2026-03-23 12:42


## 1. File Inventory

In [60]:
files = sorted(glob.glob(os.path.join(TABLES, '*.csv')))
print(f'Found {len(files)} CSV files in tables/:\n')
for f in files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {os.path.basename(f):<55} {size_mb:>6.1f} MB')

Found 13 CSV files in tables/:

  All_Subjects_ADAS_26Aug2025.csv                            1.6 MB
  All_Subjects_APOERES_26Aug2025.csv                         0.3 MB
  All_Subjects_CDR_26Aug2025.csv                             2.5 MB
  All_Subjects_DXSUM_26Aug2025.csv                           3.2 MB
  All_Subjects_GDSCALE_29Sep2025.csv                         2.5 MB
  All_Subjects_GENETIC_29Sep2025.csv                         2.7 MB
  All_Subjects_MMSE_26Aug2025.csv                            3.9 MB
  All_Subjects_NPIQ_29Sep2025.csv                            1.5 MB
  All_Subjects_PTDEMOG_26Aug2025.csv                         2.2 MB
  All_Subjects_UCD_WMH_29Sep2025.csv                         1.9 MB
  All_Subjects_UCSFFSX7_26Aug2025.csv                       30.1 MB
  All_Subjects_UPENNBIOMK_ROCHE_ELECSYS_29Sep2025.csv        0.4 MB
  baseline_subj.csv                                          0.6 MB


## 2. Load Raw Data

In [61]:
def find(pattern):
    m = glob.glob(os.path.join(TABLES, pattern))
    return m[0] if m else None

def load(pattern, label):
    p = find(pattern)
    if not p:
        print(f'  ✗ {label}: NOT FOUND ({pattern})')
        return pd.DataFrame()
    df = pd.read_csv(p, low_memory=False)
    uid = df['RID'].nunique() if 'RID' in df.columns else '?'
    print(f'  ✓ {label:<12} {len(df):>7,} rows  {uid:>5} subjects  [{os.path.basename(p)}]')
    return df

print('Loading raw data files...')
raw = {
    'demog'   : load('All_Subjects_PTDEMOG*.csv',   'DEMOG'),
    'apoe'    : load('All_Subjects_APOERES*.csv',   'APOE'),
    'mmse'    : load('All_Subjects_MMSE*.csv',      'MMSE'),
    'cdr'     : load('All_Subjects_CDR*.csv',       'CDR'),
    'adas'    : load('All_Subjects_ADAS*.csv',      'ADAS'),
    'dxsum'   : load('All_Subjects_DXSUM*.csv',     'DXSUM'),
    'gds'     : load('All_Subjects_GDSCALE*.csv',   'GDS'),
    'npiq'    : load('All_Subjects_NPIQ*.csv',      'NPIQ'),
    'ucsf'    : load('All_Subjects_UCSFFSX7*.csv',  'UCSFFSX7'),
    'baseline': load('baseline_subj.csv',           'BASELINE'),
    'genetic' : load('All_Subjects_GENETIC*.csv',   'GENETIC'),
}
print(f'\n{sum(not v.empty for v in raw.values())} / {len(raw)} files loaded.')

Loading raw data files...
  ✓ DEMOG          6,196 rows   4942 subjects  [All_Subjects_PTDEMOG_26Aug2025.csv]
  ✓ APOE           2,866 rows   2865 subjects  [All_Subjects_APOERES_26Aug2025.csv]
  ✓ MMSE          14,526 rows   4740 subjects  [All_Subjects_MMSE_26Aug2025.csv]
  ✓ CDR           14,528 rows   4391 subjects  [All_Subjects_CDR_26Aug2025.csv]
  ✓ ADAS          12,757 rows   2985 subjects  [All_Subjects_ADAS_26Aug2025.csv]
  ✓ DXSUM         15,637 rows   3787 subjects  [All_Subjects_DXSUM_26Aug2025.csv]
  ✓ GDS           13,639 rows   4584 subjects  [All_Subjects_GDSCALE_29Sep2025.csv]
  ✓ NPIQ           7,301 rows   2007 subjects  [All_Subjects_NPIQ_29Sep2025.csv]
  ✓ UCSFFSX7      11,944 rows   3127 subjects  [All_Subjects_UCSFFSX7_26Aug2025.csv]
  ✓ BASELINE       4,746 rows   4746 subjects  [baseline_subj.csv]
  ✓ GENETIC       10,035 rows   3321 subjects  [All_Subjects_GENETIC_29Sep2025.csv]

11 / 11 files loaded.


## 3. Process Each Data Source

### 3.1 Demographics

In [62]:
def process_demographics(df):
    df = df.copy()
    # Take one row per subject (demographics are static)
    if 'VISCODE2' in df.columns:
        df = df.sort_values('VISCODE2').groupby('RID').first().reset_index()
    else:
        df = df.groupby('RID').first().reset_index()
    
    keep = ['RID', 'PTGENDER', 'PTDOBYY', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT']
    df = df[[c for c in keep if c in df.columns]].copy()
    
    df['PTGENDER']  = df['PTGENDER'].replace(-4, np.nan)
    df['PTEDUCAT']  = df['PTEDUCAT'].replace(-4, np.nan)
    df['SEX']       = df['PTGENDER'].map({1: 'Male', 2: 'Female'})
    
    return df

demog = process_demographics(raw['demog'])
print(f'Demographics: {len(demog)} subjects')
print(demog[['RID','SEX','PTDOBYY','PTEDUCAT']].head(3))

Demographics: 4942 subjects
   RID     SEX  PTDOBYY  PTEDUCAT
0    1  Female   1944.0      18.0
1    2    Male   1931.0      16.0
2    3    Male   1924.0      18.0


### 3.2 APOE Genotype

In [63]:
def process_apoe(df):
    df = df[['RID', 'GENOTYPE']].drop_duplicates(subset='RID').copy()
    df['APOE4_DOSE']    = df['GENOTYPE'].apply(lambda g: str(g).count('4') if pd.notna(g) else np.nan)
    df['APOE4_CARRIER'] = (df['APOE4_DOSE'] > 0).astype('Int64')
    return df

apoe = process_apoe(raw['apoe'])
print(f'APOE: {len(apoe)} subjects')
print('\nAPOE4 dose distribution:')
print(apoe['APOE4_DOSE'].value_counts().sort_index().to_string())

APOE: 2865 subjects

APOE4 dose distribution:
APOE4_DOSE
0    1572
1    1039
2     254


### 3.3 Cognitive Measures (MMSE, CDR, ADAS)

In [64]:
def process_cog(df, score_col, label):
    df = df.copy()
    # Determine date column
    date_col = next((c for c in ['VISDATE', 'EXAMDATE'] if c in df.columns), None)
    
    viscode_col = 'VISCODE2' if 'VISCODE2' in df.columns else 'VISCODE'
    if viscode_col in df.columns:
        df['VISCODE2'] = df[viscode_col].apply(norm_viscode)
    
    keep = ['RID', 'VISCODE2', score_col]
    if date_col: keep.append(date_col)
    df = df[[c for c in keep if c in df.columns]].copy()
    
    if date_col and date_col != 'VISDATE':
        df = df.rename(columns={date_col: 'VISDATE'})
    
    df = df.dropna(subset=[score_col])
    df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
    df = df.dropna(subset=[score_col])
    df = df.drop_duplicates(subset=['RID', 'VISCODE2'], keep='first')
    
    print(f'  {label:<10}: {df["RID"].nunique():>5} subjects, {len(df):>7,} observations')
    return df

print('Cognitive measures:')
mmse = process_cog(raw['mmse'], 'MMSCORE', 'MMSE')

# CDR: prefer CDRSB
cdr_df = raw['cdr'].copy()
if 'CDRSB' not in cdr_df.columns and 'CDGLOBAL' in cdr_df.columns:
    cdr_df['CDRSB'] = cdr_df['CDGLOBAL']
cdr = process_cog(cdr_df, 'CDRSB', 'CDR-SB')

# ADAS: prefer TOTAL13, fallback to TOTSCORE
adas_df = raw['adas'].copy()
if 'TOTAL13' in adas_df.columns:
    adas_df['ADAS_COG'] = adas_df['TOTAL13']
elif 'TOTSCORE' in adas_df.columns:
    adas_df['ADAS_COG'] = adas_df['TOTSCORE']
adas = process_cog(adas_df, 'ADAS_COG', 'ADAS-Cog')

Cognitive measures:
  MMSE      :  4717 subjects,  14,380 observations
  CDR-SB    :  4345 subjects,  14,283 observations
  ADAS-Cog  :  2975 subjects,  12,466 observations


### 3.4 Diagnosis

In [65]:
DX_MAP = {1: 'CN', 2: 'MCI', 3: 'AD'}

def process_diagnosis(df):
    df = df.copy()
    viscode_col = 'VISCODE2' if 'VISCODE2' in df.columns else 'VISCODE'
    df['VISCODE2'] = df[viscode_col].apply(norm_viscode)
    
    date_col = next((c for c in ['EXAMDATE', 'VISDATE'] if c in df.columns), None)
    keep = ['RID', 'VISCODE2', 'DIAGNOSIS']
    if date_col: keep.append(date_col)
    df = df[[c for c in keep if c in df.columns]].dropna(subset=['DIAGNOSIS']).copy()
    df['DIAGNOSIS'] = df['DIAGNOSIS'].astype(int)
    df['DX_LABEL']  = df['DIAGNOSIS'].map(DX_MAP)
    df = df.drop_duplicates(subset=['RID', 'VISCODE2'], keep='first')
    if date_col and date_col in df.columns:
        df = df.rename(columns={date_col: 'VISDATE'})
    
    # Baseline diagnosis
    bl = df[df['VISCODE2'] == 'bl'].drop_duplicates(subset='RID', keep='first')[['RID', 'DIAGNOSIS', 'DX_LABEL']]
    bl = bl.rename(columns={'DIAGNOSIS': 'BL_DIAGNOSIS', 'DX_LABEL': 'BL_DX_LABEL'})
    
    print(f'Diagnosis longitudinal: {df["RID"].nunique()} subjects, {len(df):,} obs')
    print(f'Baseline diagnosis: {len(bl)} subjects')
    print(bl['BL_DX_LABEL'].value_counts().to_string())
    return df, bl

dxlong, bl_dx = process_diagnosis(raw['dxsum'])

Diagnosis longitudinal: 3776 subjects, 13,607 obs
Baseline diagnosis: 3775 subjects
BL_DX_LABEL
MCI    1606
CN     1600
AD      569


### 3.5 GDS — Depression Covariate (New — Reviewer 2)

In [66]:
def process_gds(df):
    df = df.copy()
    viscode_col = "VISCODE2" if "VISCODE2" in df.columns else "VISCODE"
    df["VISCODE2"] = df[viscode_col].apply(norm_viscode)

    if "GDTOTAL" not in df.columns:
        gds_cols = [c for c in df.columns if "GD" in c.upper() and "TOTAL" in c.upper()]
        if gds_cols:
            df["GDTOTAL"] = df[gds_cols[0]]
        else:
            print("  WARNING: GDTOTAL not found in GDS file")
            return pd.DataFrame()

    date_col = next((c for c in ["VISDATE", "EXAMDATE"] if c in df.columns), None)
    keep = ["RID", "VISCODE2", "GDTOTAL"]
    if date_col: keep.append(date_col)
    df = df[[c for c in keep if c in df.columns]].copy()

    # FIX: Replace ADNI missing code -4 with NaN (18 rows affected)
    df["GDTOTAL"] = df["GDTOTAL"].replace(-4, np.nan)

    df = df.dropna(subset=["GDTOTAL"])
    df["GDTOTAL"] = pd.to_numeric(df["GDTOTAL"], errors="coerce")
    df = df.dropna(subset=["GDTOTAL"])
    df = df.drop_duplicates(subset=["RID", "VISCODE2"], keep="first")
    if date_col and date_col in df.columns:
        df = df.rename(columns={date_col: "VISDATE"})

    print(f"GDS: {df['RID'].nunique()} subjects, {len(df):,} obs")
    print(f"     Range {df['GDTOTAL'].min():.0f}\u2013{df['GDTOTAL'].max():.0f}, mean {df['GDTOTAL'].mean():.1f}")
    return df

gds = process_gds(raw["gds"])


GDS: 4557 subjects, 13,397 obs
     Range 0–14, mean 1.7


### 3.6 NPI-Q — Neuropsychiatric Covariate (New — Reviewer 2)

In [67]:
def process_npiq(df):
    df = df.copy()
    viscode_col = 'VISCODE2' if 'VISCODE2' in df.columns else 'VISCODE'
    df['VISCODE2'] = df[viscode_col].apply(norm_viscode)
    
    # Find NPI score column
    if 'NPISCORE' not in df.columns:
        npi_cols = [c for c in df.columns if 'NPI' in c.upper() and ('SCORE' in c.upper() or 'TOTAL' in c.upper())]
        if npi_cols:
            df['NPISCORE'] = df[npi_cols[0]]
        else:
            print('  WARNING: NPISCORE not found')
            return pd.DataFrame()
    
    date_col = next((c for c in ['VISDATE', 'EXAMDATE'] if c in df.columns), None)
    keep = ['RID', 'VISCODE2', 'NPISCORE']
    if date_col: keep.append(date_col)
    df = df[[c for c in keep if c in df.columns]].copy()
    df = df.dropna(subset=['NPISCORE'])
    df['NPISCORE'] = pd.to_numeric(df['NPISCORE'], errors='coerce')
    df = df.dropna(subset=['NPISCORE'])
    df = df.drop_duplicates(subset=['RID', 'VISCODE2'], keep='first')
    if date_col and date_col in df.columns:
        df = df.rename(columns={date_col: 'VISDATE'})
    
    print(f'NPI-Q: {df["RID"].nunique()} subjects, {len(df):,} obs')
    return df

npiq = process_npiq(raw['npiq'])

NPI-Q: 2003 subjects, 7,264 obs


### 3.7 Hippocampal Volumes — Continuous (Reviewer 1 Critical Fix)
Using UCSFFSX7 FreeSurfer data: **HIPPO_TOTAL = ST29SV (L) + ST30SV (R)**, ICV = ST120SV.  
QC filter applied. Baseline visit only for cross-sectional hippocampal measure.

In [68]:
def process_imaging(ucsf_df, baseline_df=None):
    """Process hippocampal volumes as CONTINUOUS (not median-split). Addresses Reviewer 1.
    FIX: OVERALLQC values are Pass/Partial/HippoOnly/NaN — only exclude explicit Fail.
    NaN rows (11,091) have 100% valid hippocampal data — they are VALID scans without QC flags.
    """
    df = ucsf_df.copy()

    # QC filter — FIXED: only exclude rows explicitly marked Fail
    # Original bug: .str.lower() == "pass" excluded NaN rows (11,091 valid scans)
    if "OVERALLQC" in df.columns:
        n_before = len(df)
        df = df[df["OVERALLQC"] != "Fail"]   # keep Pass, Partial, HippoOnly, NaN
        n_after = len(df)
        print(f"QC filter: {n_before:,} → {n_after:,} rows (removed {n_before-n_after:,} explicit Fail)")
        qc_counts = ucsf_df["OVERALLQC"].value_counts(dropna=False)
        print(f"  QC value distribution: {dict(qc_counts.to_dict())}")

    # Standardize visit code
    viscode_col = "VISCODE2" if "VISCODE2" in df.columns else "VISCODE"
    df["VISCODE2"] = df[viscode_col].apply(norm_viscode)

    # Take baseline visit per subject
    df_bl = df[df["VISCODE2"] == "bl"].copy()
    if len(df_bl) == 0:
        print("No bl visits found, using earliest visit per subject")
        df_bl = df.sort_values("VISCODE2").groupby("RID").first().reset_index()
    df_bl = df_bl.drop_duplicates(subset="RID", keep="first")
    print(f"Baseline imaging: {len(df_bl)} subjects")

    # Hippocampal volume: L + R
    if "ST29SV" in df_bl.columns and "ST30SV" in df_bl.columns:
        df_bl["HIPPO_TOTAL"] = pd.to_numeric(df_bl["ST29SV"], errors="coerce") + \
                               pd.to_numeric(df_bl["ST30SV"], errors="coerce")
        print("HIPPO_TOTAL = ST29SV (L) + ST30SV (R)")
    elif "ST29SV" in df_bl.columns:
        df_bl["HIPPO_TOTAL"] = pd.to_numeric(df_bl["ST29SV"], errors="coerce")
        print("WARNING: Only left hippocampus found (ST29SV)")

    # ICV
    if "ST120SV" in df_bl.columns:
        df_bl["ICV"] = pd.to_numeric(df_bl["ST120SV"], errors="coerce")

    # Extract relevant columns
    keep = ["RID", "HIPPO_TOTAL", "ICV"]
    for c in ["ST101SV", "ST103CV"]:
        if c in df_bl.columns:
            keep.append(c)

    img = df_bl[[c for c in keep if c in df_bl.columns]].copy()
    img = img.dropna(subset=["HIPPO_TOTAL", "ICV"])
    img = img[img["ICV"] > 0]

    # ICV-adjusted — CONTINUOUS, not median-split (Critical: Reviewer 1.2)
    img["HIPPO_ICV_ADJ"] = img["HIPPO_TOTAL"] / img["ICV"]

    # Z-score for standardised effect interpretation
    m, s = img["HIPPO_ICV_ADJ"].mean(), img["HIPPO_ICV_ADJ"].std()
    img["HIPPO_ICV_Z"] = (img["HIPPO_ICV_ADJ"] - m) / s

    print(f"Final imaging sample: {len(img)} subjects")
    print(f"  HIPPO_TOTAL (mm3):  {img['HIPPO_TOTAL'].mean():,.0f} +/- {img['HIPPO_TOTAL'].std():,.0f}")
    print(f"  ICV (mm3):          {img['ICV'].mean():,.0f} +/- {img['ICV'].std():,.0f}")
    print(f"  HIPPO_ICV_ADJ:      {m:.4f} +/- {s:.4f}")

    return img

imaging = process_imaging(raw["ucsf"], raw["baseline"])


QC filter: 11,944 → 11,940 rows (removed 4 explicit Fail)
  QC value distribution: {nan: 11091, 'Partial': 524, 'Pass': 319, 'Hippocampus Only': 6, 'Fail': 4}
Baseline imaging: 3064 subjects
HIPPO_TOTAL = ST29SV (L) + ST30SV (R)
Final imaging sample: 3010 subjects
  HIPPO_TOTAL (mm3):  4,404 +/- 751
  ICV (mm3):          6,348 +/- 797
  HIPPO_ICV_ADJ:      0.7000 +/- 0.1267


### 3.8 Survival Variables from baseline_subj.csv
Pre-computed `Event` (CN→MCI/AD conversion) and `Time` (months to event/censoring).

In [69]:
def process_survival(df):
    df = df.copy()
    keep = ["RID"]
    for c in ["Event", "Time", "TimeMonths", "BL_DATE"]:
        if c in df.columns:
            keep.append(c)

    surv = df[keep].drop_duplicates(subset="RID", keep="first").copy()

    # FIX: Time column in baseline_subj.csv is in MONTHS (max=226 months=18.8 yrs)
    # Original bug assumed Time was already in years — caused median follow-up = 0
    if "Time" in surv.columns:
        if surv["Time"].max() > 50:   # >50 means months not years
            print(f"  Converting Time from months to years (max was {surv['Time'].max():.1f} months)")
            surv["Time_years"] = surv["Time"] / 12
        else:
            surv["Time_years"] = surv["Time"]
    elif "TimeMonths" in surv.columns:
        surv["Time_years"] = surv["TimeMonths"] / 12

    surv = surv.rename(columns={"Time_years": "TIME_YEARS"})

    print(f"Survival data: {len(surv)} subjects")
    if "Event" in surv.columns:
        print(f"  Events (conversions): {surv['Event'].sum():.0f} / {len(surv)}")
    if "TIME_YEARS" in surv.columns:
        print(f"  Follow-up (years): median={surv['TIME_YEARS'].median():.1f}, max={surv['TIME_YEARS'].max():.1f}")
    return surv

survival = process_survival(raw["baseline"])
print(survival[["RID","Event","TIME_YEARS"]].head(5).to_string())


  Converting Time from months to years (max was 226.1 months)
Survival data: 4746 subjects
  Events (conversions): 1014 / 4746
  Follow-up (years): median=0.0, max=18.8
   RID  Event  TIME_YEARS
0    1      0    0.000000
1    2      0   11.275000
2    3      1    0.119444
3    4      0    0.000000
4    5      0    0.000000


## 4. Merge All Data

In [70]:
print("Building master longitudinal dataset...")
print("Base: MMSE")

# Start with MMSE (most complete cognitive measure)
master = mmse.copy()

# FIX: LEFT merge (was outer) — preserves MMSE backbone, adds CDR/ADAS where available
# Outer merge was creating 582 extra MMSCORE-null rows that failed complete-case filter
for df, cols, label in [
    (cdr,  ["RID", "VISCODE2", "CDRSB"],    "CDR-SB"),
    (adas, ["RID", "VISCODE2", "ADAS_COG"], "ADAS"),
]:
    if not df.empty:
        master = master.merge(df[[c for c in cols if c in df.columns]], on=["RID", "VISCODE2"], how="left")
        print(f"+ {label}: {len(master):,} rows (left merge - MMSE backbone preserved)")

# Left merge diagnosis by visit
if not dxlong.empty:
    dx_cols = ["RID", "VISCODE2", "DIAGNOSIS", "DX_LABEL"]
    if "VISDATE" in dxlong.columns: dx_cols.append("VISDATE")
    master = master.merge(dxlong[[c for c in dx_cols if c in dxlong.columns]], on=["RID", "VISCODE2"], how="left")
    print(f"+ Diagnosis: {len(master):,} rows")

# Merge GDS and NPI-Q by visit
if not gds.empty:
    master = master.merge(gds[["RID", "VISCODE2", "GDTOTAL"]], on=["RID", "VISCODE2"], how="left")
    print(f"+ GDS: {len(master):,} rows")

if not npiq.empty:
    master = master.merge(npiq[["RID", "VISCODE2", "NPISCORE"]], on=["RID", "VISCODE2"], how="left")
    print(f"+ NPIQ: {len(master):,} rows")

# Merge static data by RID only
for df, label in [(demog, "Demographics"), (apoe, "APOE"), (bl_dx, "BL Diagnosis"), (imaging, "Imaging")]:
    if not df.empty:
        master = master.merge(df, on="RID", how="left")
        print(f"+ {label}: {len(master):,} rows")

# Merge survival variables
if not survival.empty:
    surv_cols = [c for c in survival.columns if c != "BL_DATE"]
    master = master.merge(survival[surv_cols], on="RID", how="left")
    print(f"+ Survival: {len(master):,} rows")

print(f"\nMaster shape: {master.shape}")
print(f"Unique subjects: {master['RID'].nunique():,}")
print(f"MMSCORE null rows: {master['MMSCORE'].isna().sum()} ({master['MMSCORE'].isna().mean()*100:.1f}%)")


Building master longitudinal dataset...
Base: MMSE
+ CDR-SB: 14,380 rows (left merge - MMSE backbone preserved)
+ ADAS: 14,380 rows (left merge - MMSE backbone preserved)
+ Diagnosis: 14,380 rows
+ GDS: 14,380 rows
+ NPIQ: 14,380 rows
+ Demographics: 14,380 rows
+ APOE: 14,380 rows
+ BL Diagnosis: 14,380 rows
+ Imaging: 14,380 rows
+ Survival: 14,380 rows

Master shape: (14380, 32)
Unique subjects: 4,717
MMSCORE null rows: 0 (0.0%)


## 5. Calculate Time Variables

In [71]:
# Consolidate VISDATE (may come from multiple merges)
visdate_cols = [c for c in master.columns if 'VISDATE' in c or 'EXAMDATE' in c]
print(f'Date columns found: {visdate_cols}')

if 'VISDATE' not in master.columns and visdate_cols:
    master['VISDATE'] = master[visdate_cols[0]]
elif 'VISDATE' in master.columns and len(visdate_cols) > 1:
    # Fill VISDATE from other date cols where missing
    for c in visdate_cols:
        if c != 'VISDATE':
            master['VISDATE'] = master['VISDATE'].combine_first(master[c])

master['VISDATE'] = pd.to_datetime(master['VISDATE'], errors='coerce')

# Baseline date = earliest visit per subject
bl_dates = master.groupby('RID')['VISDATE'].min().reset_index().rename(columns={'VISDATE': 'BL_DATE'})
master = master.merge(bl_dates, on='RID', how='left')

# Years from baseline
master['YEARS_FROM_BL'] = (master['VISDATE'] - master['BL_DATE']).dt.days / 365.25

# Age at visit
if 'PTDOBYY' in master.columns:
    master['AGE'] = master['VISDATE'].dt.year - master['PTDOBYY']

# Sort by subject then time
master = master.sort_values(['RID', 'YEARS_FROM_BL']).reset_index(drop=True)

print(f'\nTime variable summary:')
print(f'  Max follow-up: {master.groupby("RID")["YEARS_FROM_BL"].max().max():.1f} years')
print(f'  Mean follow-up: {master.groupby("RID")["YEARS_FROM_BL"].max().mean():.1f} years')
if 'AGE' in master.columns:
    print(f'  Age at baseline: {master[master["VISCODE2"]=="bl"]["AGE"].mean():.1f} ± {master[master["VISCODE2"]=="bl"]["AGE"].std():.1f} years')

Date columns found: ['VISDATE_x', 'VISDATE_y']

Time variable summary:
  Max follow-up: 19.3 years
  Mean follow-up: 2.2 years
  Age at baseline: 71.9 ± 7.8 years


## 6. Create Analysis Subsets

In [72]:
# Complete cases: MMSE + APOE + Hippocampus + Baseline Diagnosis
complete_mask = (
    master['MMSCORE'].notna() &
    master['APOE4_DOSE'].notna() &
    master['HIPPO_TOTAL'].notna() &
    master['BL_DX_LABEL'].notna()
)
complete = master[complete_mask].copy()

# Baseline only (first visit per subject in complete cases)
baseline = complete[complete['VISCODE2'] == 'bl'].drop_duplicates(subset='RID', keep='first').copy()
if len(baseline) == 0:
    # Fallback: earliest visit per subject
    baseline = complete.sort_values('YEARS_FROM_BL').groupby('RID').first().reset_index()

# Multi-visit subset (for longitudinal models)
visits_per_subj = complete.groupby('RID').size()
multi_visit_rids = visits_per_subj[visits_per_subj >= 2].index
longit = complete[complete['RID'].isin(multi_visit_rids)].copy()

print('ANALYSIS SUBSETS')
print('='*50)
print(f'Master (all):          {master["RID"].nunique():>6,} subjects, {len(master):>8,} obs')
print(f'Complete cases:        {complete["RID"].nunique():>6,} subjects, {len(complete):>8,} obs')
print(f'Baseline (complete):   {len(baseline):>6,} subjects')
print(f'Longitudinal (≥2 vis): {longit["RID"].nunique():>6,} subjects, {len(longit):>8,} obs')
print()
print('Baseline diagnosis breakdown (complete cases):')
if 'BL_DX_LABEL' in baseline.columns:
    print(baseline['BL_DX_LABEL'].value_counts().to_string())
print()
print('APOE4 dose (complete cases):')
print(baseline['APOE4_DOSE'].value_counts().sort_index().to_string())

ANALYSIS SUBSETS
Master (all):           4,717 subjects,   14,380 obs
Complete cases:         2,417 subjects,   11,793 obs
Baseline (complete):    2,417 subjects
Longitudinal (≥2 vis):  2,143 subjects,   11,519 obs

Baseline diagnosis breakdown (complete cases):
BL_DX_LABEL
MCI    1108
CN      901
AD      408

APOE4 dose (complete cases):
APOE4_DOSE
0.0    1322
1.0     870
2.0     225


## 7. Data Quality Report

In [73]:
print('MISSING DATA RATES — MASTER DATASET')
print('='*55)
key_vars = ['MMSCORE', 'CDRSB', 'ADAS_COG', 'APOE4_DOSE',
            'HIPPO_TOTAL', 'HIPPO_ICV_ADJ', 'GDTOTAL', 'NPISCORE',
            'BL_DX_LABEL', 'AGE', 'SEX', 'PTEDUCAT', 'YEARS_FROM_BL']

for v in key_vars:
    if v in master.columns:
        n_miss = master[v].isna().sum()
        pct = n_miss / len(master) * 100
        bar = '█' * int((100 - pct) / 5)
        print(f'  {v:<18} {pct:5.1f}% missing  {bar}')

print()
print('FOLLOW-UP SUMMARY — COMPLETE CASES')
print('='*55)
fu = complete.groupby('RID')['YEARS_FROM_BL'].agg(['count', 'max'])
print(f'  Mean visits/subject : {fu["count"].mean():.1f}')
print(f'  Max visits/subject  : {fu["count"].max()}')
print(f'  Mean follow-up (yrs): {fu["max"].mean():.1f}')
print(f'  Max follow-up (yrs) : {fu["max"].max():.1f}')
print(f'  Subjects with ≥2 vis: {(fu["count"] >= 2).sum()}')

MISSING DATA RATES — MASTER DATASET
  MMSCORE              0.0% missing  ████████████████████
  CDRSB                4.2% missing  ███████████████████
  ADAS_COG            13.6% missing  █████████████████
  APOE4_DOSE          13.3% missing  █████████████████
  HIPPO_TOTAL         13.8% missing  █████████████████
  HIPPO_ICV_ADJ       13.8% missing  █████████████████
  GDTOTAL              9.9% missing  ██████████████████
  NPISCORE            64.6% missing  ███████
  BL_DX_LABEL          6.7% missing  ██████████████████
  AGE                  0.5% missing  ███████████████████
  SEX                  2.9% missing  ███████████████████
  PTEDUCAT             2.9% missing  ███████████████████
  YEARS_FROM_BL        0.0% missing  ███████████████████

FOLLOW-UP SUMMARY — COMPLETE CASES
  Mean visits/subject : 4.9
  Max visits/subject  : 20
  Mean follow-up (yrs): 4.2
  Max follow-up (yrs) : 19.3
  Subjects with ≥2 vis: 2143


## 8. Save Output Files

In [74]:
outputs = {
    'ADNI_Master_Longitudinal.csv' : master,
    'ADNI_Complete_Cases.csv'      : complete,
    'ADNI_Baseline_Analysis.csv'   : baseline,
    'ADNI_Longitudinal_Analysis.csv': longit,
}

print('Saving output files to reports/...')
for fname, df in outputs.items():
    path = os.path.join(REPORTS, fname)
    df.to_csv(path, index=False)
    print(f'  ✓ {fname:<40} {len(df):>8,} rows, {df["RID"].nunique():>6,} subjects')

# Data quality report text
report_lines = [
    '='*70,
    'ADNI DATA QUALITY REPORT',
    f'Generated: {datetime.now():%Y-%m-%d %H:%M}',
    '='*70,
    f'\nMASTER DATASET',
    f'  Total observations : {len(master):,}',
    f'  Unique subjects    : {master["RID"].nunique():,}',
    f'\nCOMPLETE CASES (MMSE + APOE + Hippo + BL Dx)',
    f'  Subjects           : {complete["RID"].nunique():,}',
    f'  Observations       : {len(complete):,}',
    f'\nBASELINE ANALYSIS SAMPLE',
    f'  Subjects           : {len(baseline):,}',
]

# APOE distribution
report_lines.append('\nAPOE4 DOSE DISTRIBUTION (complete cases, baseline)')
for dose in [0, 1, 2]:
    n = (baseline['APOE4_DOSE'] == dose).sum()
    pct = n / len(baseline) * 100
    report_lines.append(f'  Dose {int(dose)}: {n} ({pct:.1f}%)')

# Baseline diagnosis
if 'BL_DX_LABEL' in baseline.columns:
    report_lines.append('\nBASELINE DIAGNOSIS DISTRIBUTION (complete cases)')
    for dx, n in baseline['BL_DX_LABEL'].value_counts().items():
        report_lines.append(f'  {dx}: {n} ({n/len(baseline)*100:.1f}%)')

report_lines.append('\nMISSING DATA (master dataset)')
for v in key_vars:
    if v in master.columns:
        n_miss = master[v].isna().sum()
        pct = n_miss / len(master) * 100
        report_lines.append(f'  {v}: {n_miss} ({pct:.1f}%) missing')

report_text = '\n'.join(report_lines)
report_path = os.path.join(REPORTS, 'ADNI_Data_Quality_Report.txt')
with open(report_path, 'w') as f:
    f.write(report_text)
print(f'\n  ✓ ADNI_Data_Quality_Report.txt')
print('\nPipeline complete!')

Saving output files to reports/...
  ✓ ADNI_Master_Longitudinal.csv               14,380 rows,  4,717 subjects
  ✓ ADNI_Complete_Cases.csv                    11,793 rows,  2,417 subjects
  ✓ ADNI_Baseline_Analysis.csv                  2,417 rows,  2,417 subjects
  ✓ ADNI_Longitudinal_Analysis.csv             11,519 rows,  2,143 subjects

  ✓ ADNI_Data_Quality_Report.txt

Pipeline complete!


## 9. Quick Sanity Check

In [75]:
print('COMPLETE CASES — COLUMN OVERVIEW')
print(complete.dtypes.to_string())
print(f'\nColumns: {list(complete.columns)}')

COMPLETE CASES — COLUMN OVERVIEW
RID                       int64
VISCODE2                 object
MMSCORE                 float64
VISDATE_x                object
CDRSB                   float64
ADAS_COG                float64
DIAGNOSIS               float64
DX_LABEL                 object
VISDATE_y                object
GDTOTAL                 float64
NPISCORE                float64
PTGENDER                float64
PTDOBYY                 float64
PTEDUCAT                float64
PTETHCAT                float64
PTRACCAT                 object
SEX                      object
GENOTYPE                 object
APOE4_DOSE              float64
APOE4_CARRIER             Int64
BL_DIAGNOSIS            float64
BL_DX_LABEL              object
HIPPO_TOTAL             float64
ICV                     float64
ST101SV                 float64
ST103CV                 float64
HIPPO_ICV_ADJ           float64
HIPPO_ICV_Z             float64
Event                     int64
Time                    float64
TimeMon

In [76]:
print('BASELINE — NUMERIC SUMMARY')
num_cols = ['AGE', 'PTEDUCAT', 'MMSCORE', 'CDRSB', 'ADAS_COG',
            'HIPPO_TOTAL', 'HIPPO_ICV_ADJ', 'GDTOTAL', 'NPISCORE']
display_cols = [c for c in num_cols if c in baseline.columns]
print(baseline[display_cols].describe().round(2).to_string())

BASELINE — NUMERIC SUMMARY
           AGE  PTEDUCAT  MMSCORE    CDRSB  ADAS_COG  HIPPO_TOTAL  HIPPO_ICV_ADJ  GDTOTAL  NPISCORE
count  2417.00   2365.00  2417.00  2417.00   2390.00      2417.00        2417.00  2416.00    938.00
mean     72.88     16.06    27.37     1.46     15.84      4443.11           0.71     1.38      1.88
std       7.37      2.74     2.67     1.78      9.56       782.73           0.13     1.47      2.82
min      50.00      4.00    15.00     0.00      0.00      1962.90           0.33     0.00      0.00
25%      68.00     14.00    26.00     0.00      8.67      3953.40           0.63     0.00      0.00
50%      73.00     16.00    28.00     1.00     14.00      4330.90           0.69     1.00      1.00
75%      78.00     18.00    29.00     2.00     21.67      4808.50           0.76     2.00      3.00
max      98.00     20.00    30.00    15.00     54.67     10128.20           1.53    12.00     24.00
